<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/notebooks/02_Metricas_basicas_de_comercio_exterior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Cuaderno 2. Métricas básicas de posición, dinamismo y apertura comercial

**Asignatura:** Inteligencia en Negocios Globales
**Semana 4 — Nivel básico**
**Documento base:** *Métricas de Comercio Exterior e Inteligencia de Negocios Globales. Documento 1 de 3 — Nivel básico: infraestructura de datos y métricas de posición comercial* (Serie de recursos, 2026)

---

## Retomamos el caso

Una empresa floricultora colombiana exporta **claveles frescos** (**HS 060312**). Hoy depende de **Estados Unidos** y quiere diversificar. El candidato sobre la mesa es **Corea del Sur**.

En el Cuaderno 1 dejamos los datos limpios y descubrimos algo incómodo: **Corea del Sur no está entre los cinco mayores importadores del mundo.** Está en el octavo lugar, con el 2,5 % del mercado mundial, frente al 26,4 % de Estados Unidos.

Si la decisión se tomara solo con el tamaño del mercado, ya estaría tomada y sería un "no". Este cuaderno existe para mostrar por qué **el tamaño del mercado es la peor variable para decidir sola**, y qué métricas hay que mirar además.

## Las tres métricas de este cuaderno

| Métrica | Pregunta que responde | Rango |
|---|---|---|
| **Balanza Comercial e IBCR** | ¿Este país produce lo que consume, o depende de comprarlo afuera? | de −1 a +1 |
| **Índice de Apertura Comercial** | ¿Qué tan atada al comercio internacional está esta economía? | porcentaje, sin techo fijo |
| **Coeficiente de Exportación** | De todo lo que producimos, ¿cuánto se va al exterior? | porcentaje de 0 a 100 |

Cada una se presenta con la misma estructura del documento base:

1. **Qué es**
2. **Para qué sirve y cómo se usa**
3. **La fórmula y su explicación matemática detallada** (el porqué de cada término, no solo la ecuación)
4. **Ejemplo numérico paso a paso**
5. **Cálculo sobre nuestros datos reales**
6. **Interpretación y umbrales**
7. **Impacto en el negocio y la decisión que habilita**
8. **De dónde se saca exactamente el dato**

## Antes de empezar

Estas métricas, advierte el documento base, "no explican causalidad; establecen las magnitudes fundamentales que describen la vulnerabilidad o resiliencia de un mercado" (Serie de recursos, 2026, p. 8, siguiendo a Durán Lima, s.f.). Son el primer filtro, no la respuesta final. Sirven para descartar rápido y para enfocar el esfuerzo analítico donde vale la pena.

---
# 0. Preparación del entorno

Este cuaderno **no parte de los datos crudos**: parte de los tres conjuntos limpios que produjo el Cuaderno 1. Si todavía no los tienes:

1. Abre el **Cuaderno 1** y ejecútalo completo.
2. Al final, la última celda descarga tres archivos: `comercio_060312_limpio.csv`, `macro_paises_limpio.csv` y `tablero_caso_limpio.csv`.
3. Vuelve aquí y súbelos cuando la celda te lo pida.

Es a propósito: los dos cuadernos son secuenciales, igual que en un proyecto real primero se prepara el dato y después se calcula.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Paleta de colores del curso (validada para lectura accesible en pantalla e impresion)
AZUL      = "#2a78d6"    # serie principal / valores positivos
ROJO      = "#e34948"    # valores negativos (par divergente con el azul)
NARANJA   = "#eb6834"    # elemento destacado
GRIS_MID  = "#c3c2b7"    # punto neutro de la escala divergente
TINTA     = "#0b0b0b"
GRIS_TEXT = "#52514e"
GRIS_EJE  = "#898781"
REJILLA   = "#e1e0d9"

print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
#  CONFIGURACION: de donde salen los datos limpios del Cuaderno 1
# ============================================================

MODO = "subir"          # opciones: "subir" | "drive" | "local"
RUTA_DRIVE = "/content/drive/MyDrive/Semana 4/datos_limpios"
RUTA_LOCAL = "datos_limpios"

if MODO == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_DATOS = RUTA_DRIVE

elif MODO == "subir":
    from google.colab import files
    print("Sube los tres archivos que genero el Cuaderno 1:")
    print("  - comercio_060312_limpio.csv")
    print("  - macro_paises_limpio.csv")
    print("  - tablero_caso_limpio.csv\n")
    files.upload()
    RUTA_DATOS = "/content"

else:
    RUTA_DATOS = RUTA_LOCAL

print("\nRuta de datos:", RUTA_DATOS)

In [ ]:
def cargar(nombre_archivo):
    '''Carga un CSV limpio del Cuaderno 1, buscandolo tambien en subcarpetas.'''
    coincidencias = sorted(glob.glob(os.path.join(RUTA_DATOS, "**", nombre_archivo), recursive=True))
    if not coincidencias:
        raise FileNotFoundError(
            f"No encontre '{nombre_archivo}' en '{RUTA_DATOS}'.\n"
            f"Ejecuta primero el Cuaderno 1 y sube los archivos que genera."
        )
    return pd.read_csv(coincidencias[0], dtype={"codigo_tm": str}, encoding="utf-8-sig")


comercio = cargar("comercio_060312_limpio.csv")
macro    = cargar("macro_paises_limpio.csv")
tablero  = cargar("tablero_caso_limpio.csv")

print(f"comercio : {comercio.shape[0]:>3} paises x {comercio.shape[1]} variables  (comercio de claveles, HS 060312, 2025)")
print(f"macro    : {macro.shape[0]:>3} paises x {macro.shape[1]} variables  (PIB y comercio total, 2024)")
print(f"tablero  : {tablero.shape[0]:>3} paises x {tablero.shape[1]} variables  (los paises del caso)")

In [ ]:
# Asi luce el tablero del caso
tablero[["pais_corto", "rol_en_el_caso", "imp_usd", "exp_usd", "imp_participacion_mundial_pct", "imp_crec_5a_pct"]]

---
# 1. Infraestructura de datos: los tres pilares clásicos

Antes de calcular nada, conviene tener claro de dónde sale cada número y qué puede y no puede responder cada fuente.

El documento base lo plantea con una frase que vale la pena memorizar: *"La validez de cualquier métrica de comercio exterior depende enteramente de la calidad y trazabilidad de su fuente: una fórmula perfecta alimentada con un dato mal definido produce una conclusión de negocio equivocada"* (Serie de recursos, 2026, p. 3).

El ecosistema de inteligencia de negocios internacionales se apoya en tres plataformas complementarias, cada una con un nivel analítico distinto (CEPAL, s.f.; Legiscomex, 2025a).

## 1.1. Trade Map (International Trade Centre)

**Qué es.** Plataforma administrada por el International Trade Centre (ITC) en cooperación con la División de Estadística de las Naciones Unidas (UNSD), construida sobre la base de datos UN Comtrade. Cubre más de 220 países y territorios y cataloga bienes en más de 5.300 clasificaciones del Sistema Armonizado (HS) a 2, 4 y 6 dígitos (ITC, 2025).

**Para qué sirve y cómo se usa.** Trade Map es la fuente que se consulta **siempre primero**, porque responde a la pregunta más básica: ¿quién exporta e importa qué, cuánto y a quién, en el mundo entero? Se usa para tres tipos de tarea:

1. Construir series históricas de exportación e importación por producto y país. Es el insumo directo de casi todas las métricas de este cuaderno: las $X_i$ y $M_i$ de las fórmulas salen de aquí.
2. Identificar mercados objetivo con el módulo *Export Potential Map*, que cruza capacidad de oferta, condiciones de demanda y condiciones de acceso al mercado.
3. Hacer comparación rápida de competidores: qué países le venden lo mismo al mismo destino y en qué volumen.

**En nuestro caso.** Los dos archivos que limpiamos en el Cuaderno 1 salieron exactamente de aquí: se seleccionó el producto HS 060312, el país reportante "Mundo", el año 2025, y se exportó a CSV.

**El Indicador de Potencial de Exportación (EPI).** Desarrollado por Decreux y Spies (2016), integra tres vectores: capacidad de oferta (rendimiento histórico del país exportador), condiciones de demanda (tamaño y crecimiento del mercado objetivo, vinculado a proyecciones de PIB del FMI) y condiciones de acceso al mercado (aranceles del *Market Access Map* del ITC y distancia geográfica como aproximación al costo logístico). El modelo asume elasticidad de oferta infinita, elasticidad de demanda de importaciones igual a uno, y sustitución de productos bajo el supuesto de Armington, es decir, bienes diferenciados según su país de origen.

**De dónde se saca el dato.** https://www.trademap.org (ITC, 2025). Metodología del EPI en Decreux y Spies (2016).

## 1.2. Legiscomex

**Qué es.** Motor de inteligencia competitiva transaccional que compila y estandariza manifiestos de carga, declaraciones de importación y exportación y documentos de transporte, con cobertura especialmente profunda en Argentina, Brasil, Chile, Colombia, Ecuador, México, Panamá, Perú y Uruguay, además de Estados Unidos, la Unión Europea y China.

**Para qué sirve y cómo se usa.** Si Trade Map responde *"cuánto se comercia"*, Legiscomex responde *"a qué precio exacto, con quién y por qué ruta"*. Se usa cuando la decisión ya no es macro (¿qué país?) sino táctica: fijar el precio de venta en un mercado específico, elegir el Incoterm y la ruta de transporte, o identificar la empresa importadora concreta a la cual dirigir una propuesta comercial.

**En nuestro caso.** Cuando la empresa haya decidido el destino, Legiscomex es la fuente para responder: ¿qué importadores coreanos de flor cortada existen, cuánto compran, a qué precio FOB entran hoy los competidores ecuatorianos y kenianos, y por qué naviera?

**De dónde se saca el dato.** https://www.legiscomex.com/informacion-estadisticas-de-comercio-exterior (Legiscomex, 2025a); metodología de inteligencia de mercados en Legiscomex (2025b).

## 1.3. EMIS (Emerging Markets Information Service)

**Qué es.** Base de inteligencia corporativa financiera con enfoque en mercados emergentes. Reporta balances, investigaciones sectoriales y noticias estructuradas, clasificando empresas según la Clasificación Industrial Internacional Uniforme (CIIU).

**Para qué sirve y cómo se usa.** EMIS se usa **antes de mirar hacia afuera**: sirve para diagnosticar si la propia industria doméstica tiene la capacidad instalada, la salud financiera y el nivel de competencia interna que justifican una estrategia de exportación agresiva. Reporta número de empresas operativas por sector, empleo generado, apalancamiento y márgenes operativos.

**En nuestro caso.** Es la fuente para responder la pregunta que ninguna base de comercio exterior puede contestar: ¿esta empresa floricultora tiene el músculo financiero y la capacidad instalada para abrir un mercado a 12.900 kilómetros de distancia?

**De dónde se saca el dato.** Plataforma EMIS (acceso institucional o universitario); documentación sectorial complementaria en fuentes CIIU nacionales.

## 1.4. Por qué ninguna reemplaza a las otras

| Fuente | Pregunta que responde | Nivel |
|---|---|---|
| **Trade Map** | ¿Existe demanda mundial y dónde crece? | Macro global |
| **Legiscomex** | ¿A qué precio y con quién se compite realmente? | Transaccional |
| **EMIS** | ¿Mi empresa o mi sector tiene la capacidad para atenderla? | Financiero doméstico |

El documento base es explícito sobre el riesgo de usar solo una: *"Un gerente que decide expandirse a un país únicamente porque Trade Map muestra crecimiento de importaciones, sin validar con EMIS si su propia industria tiene margen operativo y capacidad instalada suficiente, corre el riesgo clásico de comprometer capital en una expansión que la empresa no puede sostener operativamente"* (Serie de recursos, 2026, p. 5).

Nuestro cuaderno trabaja con el primer pilar. **Eso hay que decirlo, no darlo por sentado:** la respuesta que vamos a producir es una recomendación de mercado prioritario, no un plan de exportación completo.

---
# 2. Infraestructura ampliada: el modelo multibase de Baena-Rojas y Cano (2026)

Para decisiones de selección de mercados con más granularidad que la de los tres pilares clásicos, Baena-Rojas y Cano (2026) documentan en su técnica **IMSFEOG** (*International Market Selection for Exports of Goods*, publicada en *Global Business Review*) un total de **18 variables oficiales agrupadas en 6 factores**, cada una con su base de datos exacta.

Esta tabla es, en sí misma, un mapa de fuentes reutilizable para cualquier análisis de mercado objetivo. Los pesos entre paréntesis son el promedio de ponderaciones asignadas por un panel de 11 expertos internacionales en comercio exterior de Colombia, España y Reino Unido.

In [ ]:
IMSFEOG = pd.DataFrame([
    ("Costo",      21.4, "Precio del producto en destino",                 "Numbeo (2025a)"),
    ("Costo",      21.4, "Costo de envio internacional",                   "Geodatos (2025); World Net Logistics (2025)"),
    ("Costo",      21.4, "Costo de cumplimiento fronterizo de importacion","World Bank (2025a)"),
    ("Logistico",  18.6, "Logistics Performance Index (LPI)",              "World Bank (2025b)"),
    ("Logistico",  18.6, "Trafico portuario de contenedores",              "World Bank (2025c)"),
    ("Logistico",  18.6, "Tiempo de transito internacional",               "Sea Distances (2025)"),
    ("Comercial",  20.4, "Aranceles aduaneros",                            "OMC / WTO (2025)"),
    ("Comercial",  20.4, "Medidas proteccionistas nocivas",                "Global Trade Alert (2025)"),
    ("Comercial",  20.4, "Indice de libertad economica",                   "Heritage Foundation (2025)"),
    ("Economico",  16.4, "Indice de costo de vida",                        "Numbeo (2025b)"),
    ("Economico",  16.4, "Inflacion anual",                                "World Bank (2025d)"),
    ("Economico",  16.4, "Tasa de desempleo",                              "World Bank (2025e)"),
    ("Politico",   12.7, "Fragile States Index",                           "The Fund for Peace (2025)"),
    ("Politico",   12.7, "Reporte de riesgo (INFORM Risk Index)",          "European Commission (2025)"),
    ("Politico",   12.7, "Democracy Index",                                "Economist Intelligence (2025)"),
    ("Cultural",   10.5, "Globalization Index",                            "KOF Swiss Economic Institute (2025)"),
    ("Cultural",   10.5, "Corruption Perceptions Index",                   "Transparency International (2025)"),
    ("Cultural",   10.5, "Distancia cultural (dimensiones de Hofstede)",   "The Culture Factor Group (2025)"),
], columns=["factor", "peso_pct", "variable", "fuente"])

print("Peso de cada factor (suma de ponderaciones del panel de expertos):\n")
print(IMSFEOG.groupby("factor")["peso_pct"].first().sort_values(ascending=False).to_string())
print()
IMSFEOG

**Para qué sirve y cómo se usa.** Este modelo se usa cuando la pregunta ya no es "¿qué métrica calculo?" sino *"¿a cuál de estos 20 países candidatos debería dirigir mi estrategia de exportación este año?"*. En lugar de decidir por intuición o por cercanía geográfica —el error más común documentado por Baena-Rojas et al. (2023) en pymes exportadoras—, cada país candidato recibe un puntaje de 0 a 10 por variable, esas variables se promedian dentro de cada factor, y los seis factores se ponderan según la tabla para obtener un puntaje final comparable.

**Impacto en el negocio.** El modelo ilustra un principio metodológico central: **cada variable debe tener una fuente pública, replicable y fechada**, evitando así estimaciones discrecionales que un gerente no pueda auditar ni un competidor pueda refutar con datos. Cuando una variable no tiene fuente oficial estandarizada (por ejemplo, el precio del producto en destino), los autores recomiendan usar fuentes de mercado como aproximación, acompañadas de un análisis de sensibilidad —simulación Monte Carlo con 1.000 corridas y variabilidad del 15 %— para controlar el sesgo y poder presentar a un comité de inversión un ranking robusto ante la incertidumbre de los datos (Baena-Rojas & Cano, 2026, p. 10).

> **Dónde encaja nuestro trabajo.** Nota que ninguna de las 18 variables del modelo IMSFEOG es el tamaño del mercado. El modelo asume que ya sabes cuáles son tus países candidatos y te ayuda a ordenarlos. **Las métricas de este cuaderno son las que producen esa lista corta de candidatos.** Por eso van primero. La técnica completa se desarrolla en el Documento 3 de la serie.

---
# 3. Métrica 1 — Balanza Comercial e Índice de Balanza Comercial Relativa (IBCR)

## 3.1. Qué es

La **balanza comercial** tradicional es la diferencia monetaria entre exportaciones ($X$) e importaciones ($M$) de un país, sector o producto en un periodo dado.

El **Índice de Balanza Comercial Relativa (IBCR)** normaliza esa diferencia dividiéndola entre el comercio total, para poder comparar países, sectores o periodos de distinto tamaño sin que la escala absoluta distorsione la comparación.

## 3.2. Para qué sirve y cómo se usa

La balanza comercial en dólares absolutos es engañosa para comparar: un déficit de USD 500 millones puede ser insignificante para Estados Unidos y catastrófico para Bolivia. El IBCR resuelve esto **normalizando el resultado a un rango fijo entre −1 y +1**, lo que permite comparar directamente, por ejemplo, la posición comercial del sector textil colombiano frente al vietnamita, sin que el tamaño de las economías distorsione la lectura.

Se usa típicamente como **primer filtro** de un análisis sectorial: antes de invertir tiempo en modelos más sofisticados, se calcula el IBCR de varios mercados candidatos para descartar rápidamente aquellos que no tienen una dependencia importadora real.

**En nuestro caso**, el IBCR responde la pregunta que decide si un país es un mercado o un competidor: *¿Corea del Sur compra claveles porque no los produce, o produce los suyos y solo complementa?* Un país que produce lo que consume no es un mercado de exportación por más grande que sea.

## 3.3. La fórmula

$$IBCR_i = \frac{X_i - M_i}{X_i + M_i}$$

donde $X_i$ son las exportaciones del producto $i$ y $M_i$ las importaciones del mismo producto, en el mismo periodo y en la misma moneda.

## 3.4. Explicación matemática detallada

Vale la pena desarmar la fórmula término por término, porque cada pieza está ahí por una razón.

**El numerador $(X_i - M_i)$** es exactamente la balanza comercial tradicional: positiva si el país exporta más de lo que importa, negativa en caso contrario. Nada nuevo: es la resta de siempre.

**El denominador $(X_i + M_i)$** es el comercio total del producto: la **suma**, no la resta, de ambos flujos. Aquí está la clave. Se suma porque lo que queremos medir es *el tamaño de la actividad comercial total del país en ese producto*, para usarlo como referencia contra la cual poner la diferencia.

**Dividir uno entre otro** tiene un efecto matemático preciso: fuerza el resultado a estar **siempre entre −1 y +1**, sin importar qué tan grandes sean $X_i$ y $M_i$ en términos absolutos. Veamos por qué, con los tres casos extremos:

| Situación | Cálculo | Resultado |
|---|---|---|
| El país solo exporta ($M_i = 0$) | $\dfrac{X - 0}{X + 0} = \dfrac{X}{X}$ | $+1$ exacto |
| El país solo importa ($X_i = 0$) | $\dfrac{0 - M}{0 + M} = \dfrac{-M}{M}$ | $-1$ exacto |
| Comercio equilibrado ($X_i = M_i$) | $\dfrac{X - X}{X + X} = \dfrac{0}{2X}$ | $0$ exacto |

Y como el numerador siempre es, en valor absoluto, menor o igual que el denominador (porque $|X - M| \leq X + M$ cuando ambos son positivos), el cociente nunca puede salirse de ese rango.

**Por qué esto importa para el negocio.** El IBCR convierte una cifra en dólares —que depende del tamaño de la economía— en una **proporción comparable entre cualquier par de países o sectores**. Esa es exactamente la razón por la que se prefiere sobre la balanza comercial simple.

**Una advertencia sobre el caso $X_i = M_i = 0$.** Si un país no comercia nada del producto, el denominador es cero y la división no está definida. Matemáticamente el resultado es indeterminado, no cero. En pandas eso produce `NaN`, y así debe quedar: un país que no comercia no tiene posición comercial que medir.

## 3.5. Ejemplo numérico paso a paso

El documento base propone este ejemplo. Supóngase que en 2025 el sector de aguacate Hass de un país exportó USD 900 millones y no importó nada de ese producto. Es la situación típica de un exportador neto consolidado, similar al caso peruano documentado por Montes Ninaquispe et al. (2025), donde las exportaciones de aguacate crecieron 12,5 % anual promedio entre 2018 y 2022:

$$IBCR = \frac{900 - 0}{900 + 0} = \frac{900}{900} = 1{,}00$$

Ahora compárese con el sector de maquinaria industrial del mismo país, que en el mismo año exportó USD 120 millones pero importó USD 2.100 millones:

$$IBCR = \frac{120 - 2{.}100}{120 + 2{.}100} = \frac{-1{.}980}{2{.}220} = -0{,}89$$

El resultado es inmediato y comparable en la misma escala: el país tiene una posición exportadora casi perfecta en aguacate ($+1{,}00$) y una dependencia importadora severa en maquinaria industrial ($-0{,}89$), **pese a que en dólares absolutos el déficit de maquinaria (USD 1.980 millones) es más de dos veces el superávit de aguacate (USD 900 millones)**.

Ese contraste es toda la razón de ser del índice: los dólares absolutos habrían dicho que la maquinaria "pesa más", y el IBCR dice correctamente que la posición en aguacate es más fuerte.

Hagámoslo en Python.

In [ ]:
def ibcr(exportaciones, importaciones):
    '''Calcula el Indice de Balanza Comercial Relativa.

    IBCR = (X - M) / (X + M), acotado entre -1 y +1.
    Si el pais no comercia el producto (X + M = 0) devuelve NaN,
    porque no existe una posicion comercial que medir.
    '''
    comercio_total = exportaciones + importaciones
    if comercio_total == 0:
        return np.nan
    return (exportaciones - importaciones) / comercio_total


# Los dos casos del documento base (cifras en millones de USD)
print(f"Aguacate Hass        : IBCR = {ibcr(900, 0):+.2f}")
print(f"Maquinaria industrial: IBCR = {ibcr(120, 2100):+.2f}")
print()
print(f"Balanza del aguacate   : USD {900 - 0:>+7,} millones")
print(f"Balanza de la maquinaria: USD {120 - 2100:>+7,} millones")
print("\nEn dolares el deficit de maquinaria es mas del doble del superavit de aguacate,")
print("pero el IBCR muestra correctamente que la posicion en aguacate es la mas fuerte.")

In [ ]:
# Comprobemos los tres casos extremos de la explicacion matematica
print("Los tres casos extremos:\n")
print(f"  Solo exporta (M = 0)     : IBCR = {ibcr(500, 0):+.2f}")
print(f"  Solo importa (X = 0)     : IBCR = {ibcr(0, 500):+.2f}")
print(f"  Equilibrado (X = M)      : IBCR = {ibcr(500, 500):+.2f}")
print(f"  No comercia (X = M = 0)  : IBCR = {ibcr(0, 0)}")
print("\nY el rango se respeta aunque las cifras sean enormes o minusculas:")
print(f"  X = 1 billon, M = 3      : IBCR = {ibcr(1e12, 3):+.6f}")
print(f"  X = 3, M = 1 billon      : IBCR = {ibcr(3, 1e12):+.6f}")

## 3.6. Cálculo sobre nuestros datos reales

Ahora aplicamos la fórmula a los 139 países del conjunto de datos limpio. En pandas no hace falta un bucle: las operaciones se aplican a la columna entera de una sola vez.

In [ ]:
# Balanza comercial del producto (en dolares)
comercio["balanza_usd"] = comercio["exp_usd"] - comercio["imp_usd"]

# IBCR: la division se hace columna contra columna.
# np.where evita dividir por cero cuando el pais no comercia el producto.
comercio_total = comercio["exp_usd"] + comercio["imp_usd"]
comercio["ibcr"] = np.where(
    comercio_total > 0,
    (comercio["exp_usd"] - comercio["imp_usd"]) / comercio_total,
    np.nan,
)

print(f"Paises con IBCR calculable : {comercio['ibcr'].notna().sum()}")
print(f"Paises sin comercio del producto (IBCR indefinido): {comercio['ibcr'].isna().sum()}")
print(f"\nRango observado: de {comercio['ibcr'].min():+.4f} a {comercio['ibcr'].max():+.4f}")
print("Como esperabamos, el indice nunca se sale del intervalo [-1, +1].")

In [ ]:
# Lo mismo para el tablero del caso
tablero["balanza_usd"] = tablero["exp_usd"] - tablero["imp_usd"]
tablero["comercio_total_usd"] = tablero["exp_usd"] + tablero["imp_usd"]
tablero["ibcr"] = np.where(
    tablero["comercio_total_usd"] > 0,
    tablero["balanza_usd"] / tablero["comercio_total_usd"],
    np.nan,
)

vista = tablero[["pais_corto", "rol_en_el_caso", "exp_usd", "imp_usd", "balanza_usd", "ibcr"]].copy()
vista = vista.sort_values("ibcr")

print("BALANZA COMERCIAL E IBCR EN CLAVELES FRESCOS (HS 060312), 2025\n")
print(vista.to_string(index=False))

In [ ]:
# Grafico divergente: el IBCR tiene un cero con significado, asi que la escala
# se construye alrededor de ese cero, con un color para cada lado.

datos = tablero.sort_values("ibcr")
colores = [AZUL if v >= 0 else ROJO for v in datos["ibcr"]]

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.barh(datos["pais_corto"], datos["ibcr"], color=colores, height=0.62)

# Etiqueta directa en cada barra, del lado que no la tapa
for barra, valor in zip(barras, datos["ibcr"]):
    desplazamiento = 0.035 if valor >= 0 else -0.035
    alineacion = "left" if valor >= 0 else "right"
    ax.text(valor + desplazamiento, barra.get_y() + barra.get_height() / 2,
            f"{valor:+.2f}", va="center", ha=alineacion, fontsize=9.5, color=GRIS_TEXT)

ax.axvline(0, color=GRIS_MID, linewidth=1.4)                       # el cero es el punto neutro
ax.set_xlim(-1.35, 1.35)
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_xlabel("IBCR   (-1 = importador neto puro   |   +1 = exportador neto puro)",
              fontsize=10, color=GRIS_TEXT)
ax.set_title("Posicion comercial en claveles frescos\nIndice de Balanza Comercial Relativa, HS 060312, 2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.04,
            "En rojo: importadores netos (mercados potenciales).  En azul: exportadores netos (competidores).\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 3.7. Interpretación y umbrales

El rango va de −1 a +1 y se lee así:

| Valor del IBCR | Categoría | Lectura de negocio |
|---|---|---|
| Cercano a **+1** | Exportador neto altamente superavitario | Domina su balanza en ese producto. **Es un competidor, no un mercado** |
| Alrededor de **0** | Comercio equilibrado | Puede haber comercio intraindustrial: exporta e importa lo mismo, en variedades distintas |
| Cercano a **−1** | Importador neto estructural | **Mercado potencial**: consume lo que no produce |

Los resultados de nuestro caso son extraordinariamente claros:

- **Corea del Sur: IBCR = −1,00 exacto.** El caso extremo puro. Corea importó claveles por USD 19,3 millones y **no exportó absolutamente nada**. No hay producción local que compita con el producto importado. Desde el punto de vista de un exportador, esta es la señal más limpia que puede dar el índice.
- **Japón: −1,00 (redondeado desde −0,9997).** Prácticamente idéntico a Corea. Exportó apenas USD 13.000 frente a USD 95,7 millones importados.
- **Estados Unidos: −0,99.** También un importador neto casi puro.
- **Polonia: −0,91.** Importador neto claro, pero con algo de producción exportable (USD 3,1 millones).
- **Países Bajos: +0,17.** Aquí está la sorpresa del análisis, y la desarrollamos aparte.
- **Colombia: +1,00.** Exportador neto casi perfecto: exportó USD 337,4 millones e importó apenas USD 73.000. Colombia es, en claveles, lo que el aguacate es en el ejemplo del documento base.

## 3.8. El caso de Países Bajos: por qué el segundo mercado más grande no es un mercado

Este es el hallazgo que justifica todo el ejercicio.

En el ranking de importaciones del Cuaderno 1, **Países Bajos aparece segundo** con USD 115 millones, casi el 15 % del mercado mundial. Un analista apurado lo pondría en la lista corta de destinos.

El IBCR dice otra cosa: **+0,17**. Países Bajos **exporta más claveles de los que importa** (USD 160,8 millones exportados contra USD 115,1 millones importados). No es un consumidor final: es el **centro de redistribución** de la flor mundial. Compra flor de Colombia, Ecuador, Kenia y Etiopía, la subasta en Aalsmeer y la revende al resto de Europa.

Las consecuencias comerciales son muy concretas:

- Vender a Países Bajos **no es entrar a un mercado final**, es entrar a un canal mayorista donde el margen se reparte con el intermediario.
- Es también entrar a competir contra el propio comprador, que revende el producto aguas abajo.
- Y ya es, con casi total seguridad, uno de los destinos actuales de la empresa: allí llega buena parte de la flor colombiana.

**Ningún ranking de tamaño de mercado habría revelado esto.** Se necesitó un índice de una sola línea de aritmética. Ese es el punto del documento base cuando dice que el IBCR se usa como primer filtro.

In [ ]:
# La evidencia numerica del caso neerlandes
nl = tablero[tablero["iso3"] == "NLD"].iloc[0]

print("PAISES BAJOS - Claveles frescos (HS 060312), 2025\n")
print(f"  Importaciones : USD {nl['imp_usd']:>14,.0f}   (2do mayor importador del mundo)")
print(f"  Exportaciones : USD {nl['exp_usd']:>14,.0f}   (2do mayor exportador del mundo)")
print(f"  Balanza       : USD {nl['balanza_usd']:>+14,.0f}")
print(f"  IBCR          : {nl['ibcr']:>+18.2f}   -> exportador neto")
print("\nPor cada dolar de clavel que compra, revende {:.2f} dolares.".format(nl["exp_usd"] / nl["imp_usd"]))
print("No es un mercado de consumo: es un centro de redistribucion.")

## 3.9. Impacto en el negocio y la decisión que habilita

El documento base plantea la lectura estratégica del IBCR desde dos ángulos.

**Desde la óptica de la inversión.** Los sectores con IBCR muy negativo en países de crecimiento económico acelerado son *"mercados internos potenciales"* idóneos para la penetración de inversión extranjera directa o el establecimiento de una planta de sustitución de importaciones. Si un país importa masivamente maquinaria y su economía crece, un inversionista puede leer esa señal como oportunidad de instalar producción local.

**Desde la óptica del exportador**, que es la nuestra: un mercado con IBCR cercano a −1 es un mercado que **no puede autoabastecerse**. Su demanda tiene que cubrirse con importaciones, sí o sí. Eso hace la demanda estructuralmente más estable que la de un país que podría sustituir importaciones con producción propia si sube el precio o cambia la política comercial.

**Y en sentido inverso**, un sector con IBCR cercano a +1 y en expansión —el caso de Colombia en claveles— es candidato natural para una **estrategia de profundización exportadora**: nuevos mercados y mayor valor agregado, más que atracción de inversión sustitutiva. Traducido a nuestro caso: la recomendación del índice para Colombia es exactamente lo que la empresa quiere hacer, buscar nuevos destinos.

**La decisión concreta que este índice habilita:** de los cinco mayores importadores del mundo, **Países Bajos queda descartado como destino final** y pasa a la categoría de canal. Los otros cuatro y Corea del Sur siguen en carrera. La lista corta se redujo de seis países a cinco, con una sola división.

## 3.10. De dónde se saca exactamente el dato

$X_i$ y $M_i$ se obtienen directamente de **Trade Map** (ITC, 2025), filtrando por código HS del producto y por país o socio comercial. Alternativamente, de las estadísticas de comercio exterior de **Legiscomex** (2025a) para el detalle transaccional.

En nuestro cuaderno concreto:

| Término | Columna | Archivo original |
|---|---|---|
| $X_i$ | `exp_usd` | `exporting-countries-in-2025_060312.csv` |
| $M_i$ | `imp_usd` | `importing-countries-in-2025_060312.csv` |

Ambas ya convertidas de miles de dólares a dólares en el Cuaderno 1.

---
# 4. Métrica 2 — Índice de Apertura Comercial

## 4.1. Qué es

Mide el grado en que la dinámica económica interna de un país está atada al mercado internacional, comparando el **comercio total** (exportaciones más importaciones **de toda la economía**) contra el **tamaño de la economía** (PIB).

## 4.2. Para qué sirve y cómo se usa

Se usa como **termómetro macro de dependencia externa** antes de decidir si un país es un socio comercial estable o vulnerable a choques globales. Un analista de riesgo país lo consulta junto con la concentración sectorial para anticipar qué tan expuesta está una economía a una recesión mundial, una guerra comercial o una caída de precios de materias primas.

**En nuestro caso**, responde una pregunta muy práctica: *si vamos a montar una operación de exportación hacia este país, ¿nos vamos a encontrar con una aduana rodada y una logística que funciona, o con un país que hace poco comercio internacional y donde cada trámite será una novedad?*

## 4.3. La fórmula

$$Apertura = \frac{X + M}{PIB} \times 100$$

donde $X$ y $M$ son las exportaciones e importaciones **totales** del país —de bienes y servicios, no de un solo producto— y $PIB$ es el producto interno bruto, en la misma moneda y el mismo año.

## 4.4. Explicación matemática detallada

**El numerador $(X + M)$** suma exportaciones e importaciones totales del país. No de un solo producto, como en el IBCR, sino de toda la economía.

**Sumar en vez de restar es intencional**, y es la diferencia conceptual clave con el IBCR. Aquí no importa si el país es superavitario o deficitario: importa **cuánto de su actividad económica total pasa por la frontera, en cualquier dirección**. Un país que exporta mucho y otro que importa mucho están igual de expuestos al comercio internacional, aunque sus balanzas tengan signos opuestos.

**Dividir esa suma entre el PIB** —el valor total de todo lo que el país produce en un año— expresa qué proporción de la actividad económica nacional depende de transacciones internacionales.

**Multiplicar por 100** simplemente expresa el resultado como porcentaje en lugar de proporción decimal.

**Una diferencia importante con el IBCR: este índice no tiene techo matemático fijo.** Economías muy pequeñas y muy integradas al comercio, que reexportan grandes volúmenes, pueden superar el 150 %. No es un error de cálculo: significa que por su territorio pasa más comercio que el valor de todo lo que produce.

¿Cómo es posible pasar del 100 %? Porque el PIB mide **valor agregado** (lo que el país efectivamente produce), mientras que las exportaciones e importaciones miden **valor bruto de las transacciones**. Un país que importa un bien por 100 y lo reexporta por 105 suma 205 al numerador y solo 5 al PIB. Singapur y Países Bajos son los ejemplos clásicos.

> **Comparación con el IBCR.** El IBCR resta dos flujos del **mismo producto** y está acotado entre −1 y +1. La Apertura suma dos flujos de **toda la economía** y no tiene techo. Son preguntas distintas: el IBCR pregunta por la posición en un producto; la Apertura pregunta por la exposición de un país.

## 4.5. Ejemplo numérico paso a paso

El documento base plantea el siguiente caso. Un país con PIB de USD 350.000 millones que exportó USD 140.000 millones e importó USD 130.000 millones en el mismo año:

$$Apertura = \frac{140{.}000 + 130{.}000}{350{.}000} \times 100 = \frac{270{.}000}{350{.}000} \times 100 = 77{,}1\ \%$$

Este país tiene una apertura comercial del 77,1 %, considerada alta.

El documento agrega una advertencia que conviene subrayar: si además un analista revisa en Trade Map que el 60 % de esas exportaciones son un solo producto básico —cobre o petróleo, por ejemplo—, la lectura combinada (apertura alta más concentración de producto alta) es **la señal de alerta clásica de vulnerabilidad sistémica** ante choques de precios internacionales. Un solo indicador nunca alcanza.

In [ ]:
def apertura_comercial(exportaciones_totales, importaciones_totales, pib):
    '''Calcula el Indice de Apertura Comercial como porcentaje del PIB.

    Apertura = (X + M) / PIB x 100
    X y M son los flujos TOTALES de la economia, no los de un producto.
    '''
    if pib == 0 or pd.isna(pib):
        return np.nan
    return (exportaciones_totales + importaciones_totales) / pib * 100


# El ejemplo del documento base (cifras en millones de USD)
resultado = apertura_comercial(140_000, 130_000, 350_000)
print(f"Comercio total : USD {140_000 + 130_000:,} millones")
print(f"PIB            : USD {350_000:,} millones")
print(f"Apertura       : {resultado:.1f} %")

## 4.6. Cálculo sobre nuestros datos reales

Aquí se ve por qué en el Cuaderno 1 nos tomamos el trabajo de descargar tres indicadores del Banco Mundial y no solo el PIB. La fórmula necesita las tres cifras y las necesita **del mismo país y del mismo año**.

Recuerda la nota de trazabilidad del Cuaderno 1: los datos de comercio de claveles son de 2025 y los macroeconómicos de 2024, porque 2024 es el año más reciente con cobertura completa para todos los países del caso. El desfase se declara, no se esconde.

In [ ]:
# Sobre los 170 paises del conjunto macro
macro["apertura_pct"] = (
    (macro["exportaciones_totales_usd"] + macro["importaciones_totales_usd"])
    / macro["pib_usd"] * 100
)

# Sobre el tablero del caso
tablero["apertura_pct"] = (
    (tablero["exportaciones_totales_usd"] + tablero["importaciones_totales_usd"])
    / tablero["pib_usd"] * 100
)

vista = tablero[["pais_corto", "rol_en_el_caso", "pib_usd",
                 "exportaciones_totales_usd", "importaciones_totales_usd", "apertura_pct"]]
vista = vista.sort_values("apertura_pct", ascending=False)

print(f"INDICE DE APERTURA COMERCIAL, {int(tablero['anio'].iloc[0])}\n")
print(vista.to_string(index=False))

In [ ]:
# Verificamos el calculo a mano para Corea del Sur, paso a paso
kr = tablero[tablero["iso3"] == "KOR"].iloc[0]

x = kr["exportaciones_totales_usd"]
m = kr["importaciones_totales_usd"]
pib = kr["pib_usd"]

print(f"COREA DEL SUR, {int(kr['anio'])} - calculo paso a paso\n")
print(f"  Paso 1. Exportaciones totales (X)      : USD {x:>20,.0f}")
print(f"          Importaciones totales (M)      : USD {m:>20,.0f}")
print(f"  Paso 2. Comercio total (X + M)         : USD {x + m:>20,.0f}")
print(f"  Paso 3. PIB                            : USD {pib:>20,.0f}")
print(f"  Paso 4. Cociente (X + M) / PIB         : {(x + m) / pib:>24.4f}")
print(f"  Paso 5. Multiplicado por 100           : {(x + m) / pib * 100:>23.2f} %")

In [ ]:
# Grafico de magnitud con la referencia del 60 % marcada explicitamente
datos = tablero.sort_values("apertura_pct")
colores = [NARANJA if c == "KOR" else AZUL for c in datos["iso3"]]

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.barh(datos["pais_corto"], datos["apertura_pct"], color=colores, height=0.62)

for barra, valor in zip(barras, datos["apertura_pct"]):
    ax.text(valor + 2.5, barra.get_y() + barra.get_height() / 2,
            f"{valor:.1f} %", va="center", fontsize=9.5, color=GRIS_TEXT)

# Umbral de referencia de la literatura convencional
ax.axvline(60, color=GRIS_EJE, linewidth=1.2, linestyle="--")
ax.text(61.5, len(datos) - 0.55, "60 %: referencia de apertura alta", fontsize=8.5, color=GRIS_EJE)
ax.set_ylim(-0.65, len(datos) - 0.22)

ax.set_xlabel(f"Comercio total como porcentaje del PIB ({int(tablero['anio'].iloc[0])})",
              fontsize=10, color=GRIS_TEXT)
ax.set_title("Que tan atada al comercio internacional esta cada economia\nIndice de Apertura Comercial",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_xlim(0, datos["apertura_pct"].max() * 1.18)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.05,
            "En naranja: Corea del Sur, el destino candidato.\n"
            "Fuente: elaboracion propia con datos del Banco Mundial, World Development Indicators (World Bank, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 4.7. Interpretación y umbrales

No existe un umbral universal, pero la literatura convencional usa las siguientes referencias:

| Rango | Lectura |
|---|---|
| Por encima de **60 %** | Apertura alta |
| Entre **30 % y 60 %** | Apertura intermedia |
| Por debajo de **30 %–40 %** | Economía relativamente cerrada |

Y una condición que el documento base califica de indispensable: **este índice se lee junto con la composición de la canasta exportadora.** Una apertura alta con canasta diversificada (manufacturas, servicios) es síntoma de integración eficiente a la economía global. Una apertura alta con canasta concentrada en materias primas es síntoma de vulnerabilidad ante ciclos de precios.

Lo que dicen nuestros resultados:

- **Países Bajos: 154,0 %.** Muy por encima del 100 %, el caso de libro del país reexportador. Confirma por completo lo que ya nos había dicho el IBCR: por Países Bajos pasa mucho más comercio del que su economía produce. Dos métricas independientes, la misma conclusión.
- **Polonia: 100,4 %.** Economía profundamente integrada a las cadenas de valor europeas.
- **Corea del Sur: 84,6 %.** Apertura claramente alta, muy por encima del umbral del 60 %, y con una canasta exportadora diversificada e industrial (semiconductores, automóviles, química). Es el perfil favorable: alta apertura **sin** la vulnerabilidad de una canasta concentrada en materias primas.
- **Reino Unido: 62,8 %.** Apenas por encima del umbral.
- **Japón: 44,9 %.** Apertura intermedia. Es una economía grande cuyo mercado interno absorbe la mayor parte de lo que produce.
- **Colombia: 36,3 %.** Apertura intermedia-baja, en el rango de las economías relativamente cerradas.
- **Estados Unidos: 25,0 %.** La apertura **más baja de todo el grupo**. Y este dato merece un párrafo aparte.

## 4.8. La lectura contraintuitiva sobre Estados Unidos

Estados Unidos es el mayor importador de claveles del mundo y, al mismo tiempo, **la economía menos abierta del grupo**: apenas el 25 % de su actividad económica pasa por la frontera.

Esto no es una contradicción, es un cambio de escala. Estados Unidos importa muchísimo en términos absolutos porque su economía es enorme (USD 29,3 billones de PIB), pero en proporción a su tamaño el comercio internacional es una porción pequeña de su actividad.

**La consecuencia para el negocio es directa y va al corazón del caso.** Cuando el comercio exterior representa una fracción pequeña de la economía, las decisiones de política comercial de ese país —aranceles, cuotas, restricciones sanitarias— **le cuestan poco internamente**. Un país cuya economía depende en un 85 % del comercio internacional lo piensa dos veces antes de levantar barreras, porque el costo interno es inmediato. Uno que depende en un 25 %, no tanto.

Para una empresa que hoy concentra sus ventas en Estados Unidos, ese es exactamente el tipo de riesgo que justifica diversificar. **La empresa no está siendo caprichosa al querer salir de allí: el dato macro respalda su intuición.**

## 4.9. Impacto en el negocio y la decisión que habilita

El documento base plantea dos lecturas:

**Para una empresa que evalúa invertir o abrir una filial comercial**, un índice de apertura alto y sostenido en el tiempo es señal positiva de que ese mercado tiene *"reglas de juego rodadas para el comercio internacional (aduanas, logística, marco regulatorio)"*. Un índice bajo o en caída puede anticipar medidas proteccionistas, controles cambiarios o fricciones burocráticas que encarecen la operación.

**Para un formulador de política comercial**, la apertura combinada con la concentración de producto define directamente si conviene priorizar una política de diversificación de mercados o de diversificación de productos.

**La decisión concreta que este índice habilita en nuestro caso:** Corea del Sur, con 84,6 % de apertura, ofrece un entorno institucional y logístico maduro para el comercio internacional. Es un mercado donde la infraestructura aduanera y logística ya está construida y probada. Sumado a su IBCR de −1,00, el perfil es coherente: **una economía abierta que no produce claveles y tiene que comprarlos afuera.**

## 4.10. De dónde se saca exactamente el dato

$X$ y $M$ totales de Trade Map (ITC, 2025) o, como hicimos nosotros, de las cuentas nacionales del Banco Mundial. El PIB nominal en dólares corrientes del **Banco Mundial, indicador NY.GDP.MKTP.CD** (World Bank, 2025), o del banco central del país analizado.

| Término | Indicador del Banco Mundial | Columna en nuestros datos |
|---|---|---|
| $X$ | `NE.EXP.GNFS.CD` — Exportaciones de bienes y servicios (USD corrientes) | `exportaciones_totales_usd` |
| $M$ | `NE.IMP.GNFS.CD` — Importaciones de bienes y servicios (USD corrientes) | `importaciones_totales_usd` |
| $PIB$ | `NY.GDP.MKTP.CD` — PIB (USD corrientes) | `pib_usd` |

**Una nota metodológica que hay que declarar siempre.** Usar los tres indicadores del mismo proveedor (Banco Mundial) tiene una ventaja: las tres cifras se construyen con la misma metodología de cuentas nacionales y son consistentes entre sí. Si tomáramos el comercio de Trade Map y el PIB del Banco Mundial, estaríamos dividiendo cifras construidas con criterios distintos, y el índice quedaría contaminado por esa diferencia metodológica. **La coherencia de la fuente importa tanto como la exactitud del dato.**

---
# 5. Métrica 3 — Coeficiente de Exportación

## 5.1. Qué es

Mide qué proporción de la **producción física o del valor agregado doméstico de un bien específico** se destina a la exportación, en lugar de venderse en el mercado interno.

## 5.2. Para qué sirve y cómo se usa

Mientras el Índice de Apertura mide la dependencia externa de **toda la economía**, el Coeficiente de Exportación la mide a nivel de **un solo sector o producto**. Es la pregunta que se hace un gerente de planta o un director comercial:

> *"De todo lo que produzco, ¿qué tanto realmente se está yendo al exterior?"*

Se usa para decidir si conviene rediseñar una línea de producción bajo estándares de exportación —empaque, certificaciones, escalabilidad— o si el negocio internacional sigue siendo marginal frente al mercado doméstico.

**En nuestro caso**, cambia el sujeto del análisis. Las dos métricas anteriores miraban hacia afuera, a los mercados candidatos. Esta mira **hacia adentro**, a la empresa y al sector colombiano. Y la respuesta va a decidir qué tipo de estrategia corresponde.

## 5.3. La fórmula

$$CE_{ij} = \left(\frac{X_{ij}}{VP_{ij}}\right) \times 100$$

## 5.4. Explicación matemática detallada

**$X_{ij}$** es el valor exportado del bien $j$ por el país (o la empresa) $i$.

**$VP_{ij}$** es el valor de **toda la producción** (o el valor agregado) de ese mismo bien en el mismo país o empresa: lo exportado **más** lo vendido domésticamente.

**El cociente entre ambos, multiplicado por 100**, responde directamente qué porcentaje de la producción total cruzó la frontera.

**Nótese la diferencia estructural con las dos métricas anteriores:**

| Métrica | Qué compara | Naturaleza de los términos |
|---|---|---|
| **IBCR** | Exportaciones contra importaciones del mismo producto | Dos flujos externos |
| **Apertura** | Comercio total contra PIB | Flujos externos contra actividad total del país |
| **Coeficiente de Exportación** | Exportaciones contra producción total del producto | Un flujo externo contra la actividad interna del sector |

Son preguntas distintas y se usan en momentos distintos del análisis.

**La consecuencia práctica más importante:** el numerador sale de una base de comercio exterior (Trade Map), pero **el denominador no**. $VP_{ij}$ sale de las cuentas nacionales o de encuestas de producción del instituto estadístico del país. Es un dato de otra naturaleza, con otra periodicidad y otra metodología. Volveremos sobre esto, porque es la principal dificultad práctica de esta métrica.

**Rango.** El resultado va de 0 % a 100 %, siempre que $VP$ esté bien medido e incluya lo exportado. Un valor superior a 100 % es señal de un problema de medición: normalmente indica que el denominador no incluye toda la producción, que hay reexportación de producto importado, o que las dos cifras corresponden a periodos distintos.

## 5.5. Ejemplo numérico paso a paso

El documento base propone el siguiente caso. Una planta de envases plásticos produce en un año el equivalente a USD 40 millones en valor agregado, de los cuales USD 26 millones se exportan y el resto se vende en el mercado local:

$$CE = \left(\frac{26}{40}\right) \times 100 = 65\ \%$$

Un coeficiente de exportación del 65 % indica que esta planta **ya diseña y opera bajo una lógica de estándares internacionales**. Es muy distinto de una segunda planta del mismo sector con $CE = 8\ \%$, cuyo negocio principal es doméstico y para la cual la exportación es apenas una salida ocasional de excedentes.

In [ ]:
def coeficiente_exportacion(exportaciones, valor_produccion):
    '''Calcula el Coeficiente de Exportacion como porcentaje de la produccion.

    CE = (X / VP) x 100
    VP es el valor total de la produccion del bien: lo exportado mas lo vendido internamente.
    '''
    if valor_produccion == 0 or pd.isna(valor_produccion):
        return np.nan
    return exportaciones / valor_produccion * 100


# El ejemplo del documento base (millones de USD)
print(f"Planta A: CE = {coeficiente_exportacion(26, 40):.1f} %   -> opera bajo estandares internacionales")
print(f"Planta B: CE = {coeficiente_exportacion(3.2, 40):.1f} %   -> el exterior es una valvula de escape")

## 5.6. Aplicación al caso colombiano, con un supuesto declarado

Aquí nos topamos con una limitación real, y la forma de manejarla es en sí misma parte de la lección.

**Tenemos el numerador.** Colombia exportó USD 337,4 millones de claveles frescos en 2025, dato de Trade Map. Es una cifra oficial y trazable.

**No tenemos el denominador.** El valor de la producción colombiana de claveles no está en ninguna base de comercio exterior. Está en las cuentas nacionales del **DANE** o en las estadísticas agropecuarias de **Agronet (Evaluaciones Agropecuarias Municipales)**, que no forman parte de los datos de esta semana.

Hay dos maneras de reaccionar ante esto:

1. **La mala:** inventar un número que suene razonable, calcular el índice y presentarlo como si fuera un resultado. Es la práctica que Baena-Rojas y Cano (2026) califican de "estimación discrecional que un gerente no puede auditar".
2. **La correcta:** trabajar con un **supuesto explícito**, calcular el índice bajo ese supuesto, y acompañarlo de un **análisis de sensibilidad** que muestre cómo cambiaría el resultado si el supuesto estuviera equivocado. Es exactamente lo que los autores recomiendan cuando una variable no tiene fuente oficial estandarizada.

Tomamos la segunda vía.

> **Supuesto didáctico declarado.** Asumimos un valor de producción de claveles en Colombia de **USD 420 millones** para 2025. Este número **no es un dato oficial**: es un supuesto de trabajo para ilustrar la métrica, construido bajo la premisa de que la floricultura colombiana de clavel es una industria fuertemente orientada a la exportación. **Antes de usar este resultado en una decisión real, hay que reemplazarlo por la cifra del DANE o de Agronet.**

In [ ]:
# ============================================================
#  SUPUESTO DECLARADO - reemplazar por el dato oficial del DANE / Agronet
# ============================================================
VALOR_PRODUCCION_SUPUESTO_USD = 420_000_000
FUENTE_SUPUESTO = "Supuesto didactico. NO es un dato oficial."
# ============================================================

exportaciones_colombia = float(tablero.loc[tablero["iso3"] == "COL", "exp_usd"].iloc[0])

ce_colombia = coeficiente_exportacion(exportaciones_colombia, VALOR_PRODUCCION_SUPUESTO_USD)

print("COEFICIENTE DE EXPORTACION - Claveles frescos, Colombia\n")
print(f"  X  (exportaciones, Trade Map 2025)   : USD {exportaciones_colombia:>15,.0f}   [dato oficial]")
print(f"  VP (valor de produccion)             : USD {VALOR_PRODUCCION_SUPUESTO_USD:>15,.0f}   [{FUENTE_SUPUESTO}]")
print(f"\n  Paso 1. Cociente X / VP              : {exportaciones_colombia / VALOR_PRODUCCION_SUPUESTO_USD:>19.4f}")
print(f"  Paso 2. Multiplicado por 100         : {ce_colombia:>18.1f} %")
print(f"\n  Venta domestica implicita            : USD {VALOR_PRODUCCION_SUPUESTO_USD - exportaciones_colombia:>15,.0f}")

## 5.7. Análisis de sensibilidad: qué pasa si el supuesto está equivocado

Un resultado que depende de un supuesto **debe presentarse junto con el rango de resultados posibles**. Así, quien tome la decisión ve de inmediato si la conclusión cambia o no cuando el supuesto se mueve.

Calculamos el coeficiente para distintos valores de producción alrededor del supuesto.

In [ ]:
# Recorremos valores de produccion desde 360 hasta 600 millones
valores_vp = np.arange(360, 620, 20) * 1_000_000

sensibilidad = pd.DataFrame({
    "valor_produccion_musd": valores_vp / 1e6,
    "coeficiente_exportacion_pct": [
        coeficiente_exportacion(exportaciones_colombia, vp) for vp in valores_vp
    ],
})
sensibilidad["supera_umbral_50"] = np.where(
    sensibilidad["coeficiente_exportacion_pct"] > 50, "Si", "No"
)

print("ANALISIS DE SENSIBILIDAD DEL COEFICIENTE DE EXPORTACION\n")
print(sensibilidad.to_string(index=False))

minimo = sensibilidad["coeficiente_exportacion_pct"].min()
print(f"\nEn TODO el rango explorado el coeficiente se mantiene por encima del 50 %")
print(f"(el valor mas bajo es {minimo:.1f} %).")
print("La conclusion cualitativa NO depende del supuesto: es robusta.")

Este es el punto del análisis de sensibilidad. El **valor exacto** del coeficiente sí depende del supuesto: puede estar entre el 56 % y el 94 % según qué valor de producción se use.

Pero la **conclusión cualitativa no cambia en ningún escenario del rango**: el coeficiente supera el 50 % siempre. Y como veremos enseguida, es el umbral del 50 % el que determina qué estrategia corresponde. **La recomendación de negocio es robusta ante la incertidumbre del dato.**

Poder decir esa frase, y respaldarla con la tabla, es lo que permite llevar un resultado incompleto a un comité de inversión sin perder credibilidad.

## 5.8. Interpretación y umbrales

| Rango del CE | Lectura |
|---|---|
| Superior al **50 %** | Industria diseñada bajo estándares internacionales: procesos, certificaciones y escala pensados para el comprador externo |
| Entre **20 % y 50 %** | Industria mixta, con presencia exportadora consolidada pero mercado interno relevante |
| Por debajo de **15 %–20 %** | El mercado exterior es solo una válvula de escape para excedentes esporádicos, sin estrategia de internacionalización consolidada |

Bajo nuestro supuesto, la floricultura colombiana de clavel se ubica claramente en la primera categoría. Y no es sorprendente: los datos del Cuaderno 1 lo respaldan por otra vía completamente independiente. **Colombia concentra el 50,2 % de las exportaciones mundiales de claveles frescos** y su IBCR es +1,00. Una industria que abastece la mitad del mercado mundial de un producto y casi no importa ese producto es, necesariamente, una industria orientada a la exportación.

## 5.9. Impacto en el negocio y la decisión que habilita

Aquí es donde las tres métricas se juntan y el caso se cierra.

El documento base es preciso sobre lo que implica cada nivel del coeficiente:

- Una empresa con **CE bajo pero demanda externa creciente** (verificable con Trade Map) es candidata a una **expansión de planta** dedicada a exportación.
- Una empresa con **CE ya alto** necesita, en cambio, **estrategias de diversificación de mercados** más que de aumento de volumen, *"porque ya depende estructuralmente del comercio exterior y un choque de demanda externa la golpearía con fuerza"*.

Nuestra empresa está en el segundo caso, y de forma inequívoca. Su coeficiente de exportación es alto en todos los escenarios. **La métrica dice, con toda claridad, que lo que necesita no es producir más, sino vender en más lugares.**

Y eso es exactamente lo que la empresa vino a preguntar. La métrica no solo valida la pregunta: valida la estrategia detrás de la pregunta. Aumentar la capacidad instalada para venderle más a Estados Unidos sería la decisión equivocada; abrir un mercado nuevo es la correcta.

## 5.10. De dónde se saca exactamente el dato

$X_{ij}$ de **Trade Map** o **Legiscomex**.

$VP_{ij}$ (valor de producción o valor agregado sectorial) de las **cuentas nacionales del DANE**, el INEGI, el INE u otro instituto estadístico nacional equivalente, o de **EMIS** para el detalle por empresa o sector bajo clasificación CIIU.

Para el caso colombiano concreto, las fuentes a consultar son:

| Dato | Fuente | Nota |
|---|---|---|
| $X_{ij}$ | Trade Map, HS 060312, reportante Colombia | Ya lo tenemos: USD 337,4 millones |
| $VP_{ij}$ sectorial | DANE, Cuentas nacionales, rama "cultivo de flores" | Nivel de agregación más amplio que el clavel solo |
| $VP_{ij}$ por producto | Agronet, Evaluaciones Agropecuarias Municipales | Producción física; requiere convertir a valor |
| $VP_{ij}$ por empresa | EMIS, código CIIU 0119 | Para el análisis de una empresa específica |

**Advertencia de coherencia.** Al reemplazar el supuesto, verifica que el $VP$ y el $X$ correspondan al **mismo año**, al **mismo nivel de producto** y a la **misma definición** (valor de producción bruto o valor agregado, pero no uno mezclado con otro). Es el mismo cuidado que tuvimos en el Cuaderno 1 al elegir un año de referencia común.

---
# 6. Lectura conjunta: el tablero de decisión

Ninguna de las tres métricas decide sola. El documento base insiste en que se leen en conjunto, y ahora vamos a hacerlo.

Construimos un tablero único con todas las variables y agregamos dos que vienen de Trade Map y aportan contexto comercial:

- **Valor unitario (USD por tonelada):** cuánto paga cada mercado por la misma flor. Es una aproximación al posicionamiento de precio y, por lo tanto, al margen posible.
- **Crecimiento de las importaciones a 5 años:** hacia dónde va el mercado, no dónde está hoy. Un mercado grande que cae y uno mediano que crece pueden cruzarse en pocos años.
- **Distancia (km):** aproximación al costo logístico, y variable crítica en un producto perecedero.

In [ ]:
TABLERO_FINAL = tablero[[
    "pais_corto", "rol_en_el_caso",
    "imp_usd", "imp_participacion_mundial_pct", "ibcr", "apertura_pct",
    "imp_crec_5a_pct", "imp_valor_unitario_usd_ton", "imp_distancia_km",
]].copy()

TABLERO_FINAL.columns = [
    "Pais", "Rol",
    "Importaciones USD", "Cuota mundial %", "IBCR", "Apertura %",
    "Crec. 5 anios %", "Precio USD/ton", "Distancia km",
]

TABLERO_FINAL = TABLERO_FINAL.sort_values("Importaciones USD", ascending=False).reset_index(drop=True)

print("TABLERO DE DECISION - Claveles frescos (HS 060312)\n")
print(TABLERO_FINAL.to_string(index=False))
print("\nComercio: Trade Map 2025. Macroeconomia: Banco Mundial 2024.")

In [ ]:
# Grafico de posicionamiento: tamano del mercado frente a su ritmo de crecimiento.
# Dos preguntas distintas en los dos ejes; el cuadrante superior derecho es el objetivo.

datos = tablero[tablero["rol_en_el_caso"] != "Pais de origen"].copy()

fig, ax = plt.subplots(figsize=(9.5, 6))

for _, fila in datos.iterrows():
    destacado = fila["iso3"] == "KOR"
    ax.scatter(fila["imp_usd"] / 1e6, fila["imp_crec_5a_pct"],
               s=260 if destacado else 190,
               color=NARANJA if destacado else AZUL,
               edgecolor="#fcfcfb", linewidth=2, zorder=3)
    ax.annotate(fila["pais_corto"],
                (fila["imp_usd"] / 1e6, fila["imp_crec_5a_pct"]),
                textcoords="offset points", xytext=(0, 15),
                ha="center", fontsize=9.5,
                color=TINTA if destacado else GRIS_TEXT,
                fontweight="bold" if destacado else "normal")

# Fijamos los limites antes de colocar las anotaciones, para que nada se salga del lienzo
ax.set_xlim(-18, datos["imp_usd"].max() / 1e6 * 1.18)
ax.set_ylim(datos["imp_crec_5a_pct"].min() - 6, datos["imp_crec_5a_pct"].max() + 6)

ax.axhline(0, color=GRIS_MID, linewidth=1.3)
ax.text(ax.get_xlim()[0] + 2, 1.4, "mercados en crecimiento",
        fontsize=8.5, color=GRIS_EJE, ha="left")
ax.text(ax.get_xlim()[0] + 2, -2.8, "mercados en contraccion",
        fontsize=8.5, color=GRIS_EJE, ha="left")

ax.set_xlabel("Tamano del mercado: importaciones 2025 (millones de USD)", fontsize=10, color=GRIS_TEXT)
ax.set_ylabel("Dinamismo: crecimiento anual de las importaciones\nen los ultimos 5 anios (%)",
              fontsize=10, color=GRIS_TEXT)
ax.set_title("Tamano no es lo mismo que oportunidad\nMercados de claveles frescos, HS 060312",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)
for lado in ["left", "bottom"]:
    ax.spines[lado].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.03,
            "En naranja: Corea del Sur, el destino candidato.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 6.1. Lectura del tablero, país por país

**Estados Unidos** — El mercado más grande del mundo, con el 26,4 % de las importaciones, IBCR de −0,99 y crecimiento de 8 % anual. Es un buen mercado. Pero es donde la empresa ya está y de donde quiere reducir su exposición. Además, su apertura comercial del 25 % es la más baja del grupo, lo que significa que las decisiones de política comercial le cuestan poco internamente. Y paga los precios más bajos, aunque el dato no es directamente comparable porque reporta en unidades y no en toneladas.

**Países Bajos** — Descartado como destino final por el IBCR (+0,17) y confirmado por la apertura de 154 %. No es un mercado: es el canal mayorista europeo. Vender allí es válido, pero es otro modelo de negocio y con otro margen.

**Japón** — Tercer mercado mundial, IBCR de −1,00, y **el precio unitario más alto del grupo: USD 8.449 por tonelada**. Es un mercado premium y estructuralmente importador. El problema está en la tendencia: sus importaciones **caen 2 % anual** desde hace cinco años. Es un mercado maduro y en contracción.

**Polonia** — El hallazgo lateral del análisis. Cuarto mercado, **crece 15 % anual** y su apertura del 100 % lo señala como plenamente integrado a las cadenas europeas. IBCR de −0,91. Su precio unitario es el más bajo entre los mercados comparables (USD 5.803 por tonelada), lo que sugiere un mercado sensible al precio. **Merece un análisis propio que este cuaderno no alcanza a hacer.**

**Reino Unido** — Quinto mercado, IBCR de −1,00, apertura de 62,8 %. Pero sus importaciones **caen 4 % anual**, la contracción más pronunciada del grupo. Es el mercado con el perfil menos atractivo de los cinco grandes.

**Corea del Sur** — El candidato. Vamos a mirarlo con detalle.

## 6.2. El caso de Corea del Sur

| Variable | Valor | Lectura |
|---|---|---|
| Importaciones | USD 19,3 millones | Mercado mediano: octavo del mundo |
| Cuota mundial | 2,50 % | Pequeño en términos relativos |
| **IBCR** | **−1,00 exacto** | **Importador neto puro: cero producción local competidora** |
| **Apertura comercial** | **84,6 %** | **Economía muy abierta, con canasta diversificada** |
| **Crecimiento a 5 años** | **+15 % anual** | **Mercado en expansión sostenida** |
| Precio unitario | USD 7.720 por tonelada | Segundo más alto del grupo, después de Japón |
| Distancia | 12.912 km | La mayor de todo el grupo |

**Los argumentos a favor**

1. **IBCR de −1,00 exacto.** No es una aproximación: Corea del Sur no exportó ni un dólar de claveles en 2025. Toda su demanda se cubre con importaciones y no existe un productor local que pueda desplazar al proveedor extranjero si sube el precio.
2. **Crece al 15 % anual**, empatado con Polonia como el mercado más dinámico del grupo y muy por delante de Estados Unidos (8 %). Japón y Reino Unido, en cambio, se contraen.
3. **Paga bien: USD 7.720 por tonelada**, un 33 % más que Polonia y un 18 % más que Países Bajos. Es un mercado que valora la calidad, no solo el precio.
4. **Apertura del 84,6 %** con canasta exportadora industrial y diversificada. Aduana rodada, logística construida, marco regulatorio de comercio internacional maduro.
5. **Diversifica el riesgo geográfico y político.** Sale de la órbita comercial estadounidense y entra a la asiática. Ese era, desde el principio, el objetivo del ejercicio.

**Los argumentos en contra, que también hay que decir**

1. **12.912 kilómetros de distancia**, la mayor del grupo. En un producto perecedero eso significa flete aéreo, cadena de frío ininterrumpida y un costo logístico que puede comerse el sobreprecio del punto 3. **Esta es la variable que puede tumbar la decisión**, y este cuaderno no la puede cuantificar: requiere el factor Logístico y el factor Costo del modelo IMSFEOG (Baena-Rojas & Cano, 2026), que se desarrollan en el Documento 3.
2. **Mercado pequeño en términos absolutos.** USD 19,3 millones es un décimo de Estados Unidos. Para una empresa grande puede no justificar el costo fijo de abrir el mercado.
3. **Alta concentración de proveedores.** El índice de concentración que reporta Trade Map para Corea (0,77) es de los más altos del grupo, lo que sugiere que pocos orígenes dominan ese mercado. Entrar significa desplazar a un proveedor establecido, no ocupar un espacio vacío. El tratamiento formal de esta variable —el índice Herfindahl-Hirschman— corresponde al Documento 2 de la serie.

## 6.3. La respuesta al caso

**Corea del Sur pasa el filtro de las métricas básicas y merece pasar a la siguiente fase del análisis.** Ese es el veredicto que estas tres métricas habilitan, y no más que eso.

Corea es un mercado mediano pero **estructuralmente importador, en crecimiento sostenido, que paga precios altos y con una economía institucionalmente abierta al comercio**. Combina las tres características que un exportador busca cuando quiere diversificar. Es una lista corta razonable.

Ahora bien, **la decisión de exportar no está tomada**, y presentarla como tomada sería exceder lo que los datos permiten. Faltan tres cosas, en este orden:

1. **Cuantificar el costo logístico.** Es el riesgo principal de la decisión y el cuaderno no lo resuelve. Requiere el factor Logístico y el factor Costo del modelo IMSFEOG.
2. **Comparar Corea contra Polonia con el mismo rigor.** Polonia crece igual de rápido, está cuatro veces más cerca de Colombia en términos de flete desde el punto de vista de la ruta europea ya establecida, y es un mercado tres veces más grande. Es un candidato serio que apareció como hallazgo lateral de este análisis y que no estaba en la pregunta original.
3. **Verificar la premisa de partida.** La empresa dice depender de Estados Unidos, y las métricas macro respaldan que esa dependencia es riesgosa. Pero **con estos datos no podemos medir cuánto depende**: los archivos de Trade Map que descargamos son de cada país contra el mundo, no bilaterales. Para cuantificar la dependencia hay que hacer una consulta distinta en Trade Map: Colombia como reportante, desglosada por país socio. Es el primer dato que hay que traer a la siguiente sesión.

> **La conclusión honesta de un análisis de nivel básico es una lista corta y una agenda de trabajo, no una decisión de inversión.** Presentarlo así, y no como una certeza, es lo que hace que un comité confíe en el analista la próxima vez.

---
# 7. Los límites de estas métricas y el puente al Documento 2

El documento base cierra con una advertencia que conviene tomar en serio: estas métricas básicas comparten todas la misma limitación. Son **estáticas** —retratan un año— y **univariadas** —analizan un producto o un país a la vez.

Concretamente, en nuestro análisis:

| Límite | Qué no pudimos responder |
|---|---|
| **Es estático** | El IBCR de Corea es −1,00 en 2025. ¿Lo era hace cinco años? ¿Y qué pasó durante la pandemia? Una serie de tiempo contaría otra historia |
| **Es univariado** | Miramos solo el HS 060312. Si el negocio incluye rosas o crisantemos, hay que repetir todo el análisis por producto |
| **No mide competencia** | Sabemos que Corea importa, no de quién. ¿Quién es el proveedor a desplazar y qué tan atrincherado está? |
| **No mide concentración formalmente** | Usamos el índice que reporta Trade Map, pero no lo calculamos ni lo interpretamos con rigor |
| **No mide competitividad** | Colombia exporta la mitad de los claveles del mundo, ¿pero eso es ventaja comparativa real o solo volumen heredado? |
| **No mide causalidad** | Nada de lo que calculamos explica *por qué* Corea importa más cada año |

**Contraste metodológico.** La Comisión Económica para América Latina y el Caribe documenta formalmente esta misma familia de indicadores básicos en su serie *Indicadores de comercio exterior y política comercial* (Durán Lima, s.f.), coincidiendo en la definición de balanza comercial relativa y apertura comercial aquí presentadas. Esa coincidencia confirma la vigencia y la estandarización académica de estas métricas más de dos décadas después de su formalización inicial. No estamos usando fórmulas caseras: son los indicadores estándar del análisis de comercio exterior.

**Lo que viene.** Cuando la pregunta de negocio requiere comparar la posición de un país frente al resto del mundo simultáneamente en muchos productos, o distinguir si el comercio de un sector es realmente competitivo o solo grande en volumen, hay que avanzar hacia:

- **Documento 2 — Concentración y ventaja comparativa.** El índice Herfindahl-Hirschman (HHI) para medir formalmente la concentración de mercados y de proveedores, el índice de Grubel-Lloyd para el comercio intraindustrial, y la familia de índices de ventaja comparativa revelada.
- **Documento 3 — Dinámica competitiva y selección de mercados.** El análisis *Constant Market Share*, la econometría gravitacional (PPML) y la técnica multicriterio IMSFEOG completa, con los seis factores y las 18 variables de la tabla de la sección 2.

Ahí es donde se responden las preguntas que este cuaderno dejó abiertas.

---
# 8. Glosario mínimo

- **HS (Sistema Armonizado):** nomenclatura internacional de clasificación de mercancías administrada por la Organización Mundial de Aduanas, usada por Trade Map, Legiscomex y la OMC para identificar productos de forma estandarizada a nivel mundial. Nuestro producto, **060312**, corresponde a claveles frescos cortados para ramos o adornos.
- **FOB / CIF:** Incoterms que definen si el precio de transacción incluye (CIF: *Cost, Insurance and Freight*) o no (FOB: *Free On Board*) el flete y el seguro internacional hasta el puerto de destino. Es una de las razones por las que las importaciones y las exportaciones mundiales de un mismo producto no cuadran exactamente.
- **CIIU:** Clasificación Industrial Internacional Uniforme, usada por EMIS y las cuentas nacionales para segmentar empresas por actividad económica.
- **Comercio total:** suma (no diferencia) de exportaciones más importaciones. Es el denominador común de casi todas las métricas normalizadas de este documento.
- **Importador neto:** país cuyas importaciones de un producto superan sus exportaciones del mismo producto. Su IBCR es negativo.
- **Reexportación:** práctica de importar un bien para revenderlo a un tercer país sin transformación sustancial. Es lo que hace que la apertura comercial de un país pueda superar el 100 %.
- **Valor unitario:** valor total dividido entre la cantidad. Solo es comparable entre países que reportan la cantidad en la misma unidad de medida.

---
# 9. Ejercicios propuestos

**Ejercicio 1 — Cambiar el candidato.** En la sección 6, reemplaza Corea del Sur por Polonia como destino candidato y reescribe la sección 6.2 con los datos de Polonia. ¿Cambia la recomendación? ¿Qué variable pesa más en el cambio?

**Ejercicio 2 — El umbral del IBCR.** Define en código un umbral: un país es "mercado potencial" si su IBCR es menor que −0,80. Filtra el conjunto `comercio` completo con ese criterio y ordena por importaciones. ¿Cuántos mercados potenciales hay en el mundo? ¿Aparece alguno que no habíamos considerado?

**Ejercicio 3 — Apertura y tamaño.** Calcula la correlación entre el PIB y el índice de apertura comercial en el conjunto `macro` completo (170 países). ¿Las economías más grandes tienden a ser más abiertas o más cerradas? Explica el resultado con lo que aprendiste sobre Estados Unidos en la sección 4.8.

**Ejercicio 4 — El supuesto del Coeficiente de Exportación.** Busca en Agronet o en el DANE una cifra real de producción de flores en Colombia, reemplaza `VALOR_PRODUCCION_SUPUESTO_USD` y vuelve a correr la sección 5. Documenta la fuente en formato APA 7. ¿Se mantiene la conclusión?

**Ejercicio 5 — Verificar la premisa.** Descarga de Trade Map la tabla bilateral de exportaciones de Colombia del HS 060312 por país de destino. Calcula qué porcentaje va a Estados Unidos. ¿La premisa de la empresa está respaldada por los datos?

**Ejercicio 6 — Un producto distinto.** Repite los dos cuadernos completos con el HS 060311 (rosas frescas). El código no necesita cambios salvo el nombre de los archivos. Esa es la ventaja de haber escrito funciones reutilizables en el Cuaderno 1.

---
# Referencias

Baena-Rojas, J. J., & Cano, J. A. (2026). International market selection for exports of goods: A data analysis technique for organizational decision-making. *Global Business Review*. https://doi.org/10.1177/09721509261464305

Baena-Rojas, J. J., Mackenzie-Torres, T., Cuesta-Giraldo, G., & Tabares, A. (2023). A hybrid multi-criteria decision-making technique for international market selection in SMEs. *Polish Journal of Management Studies, 27*(1), 26–45. https://doi.org/10.17512/pjms.2023.27.1.02

Comisión Económica para América Latina y el Caribe. (s.f.). *Indicadores de comercio exterior y política comercial*. CEPAL.

Decreux, Y., & Spies, J. (2016). *Export potential and diversification assessment: A methodology to identify export opportunities*. International Trade Centre. https://umbraco.exportpotential.intracen.org/media/cklh2pi5/epa-methodology_230627.pdf

Departamento Administrativo Nacional de Estadística. (2025). *Cuentas nacionales*. https://www.dane.gov.co/index.php/estadisticas-por-tema/cuentas-nacionales

Durán Lima, J. E. (s.f.). *Indicadores de comercio exterior y política comercial: Generalidades metodológicas e indicadores básicos*. Comisión Económica para América Latina y el Caribe. https://repositorio.cepal.org/server/api/core/bitstreams/7fb05ad7-2ee7-474b-9382-2e9416e5a9a5/content

Google. (2025). *Colaboratory: Preguntas frecuentes*. https://research.google.com/colaboratory/faq.html

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E. (2020). Array programming with NumPy. *Nature, 585*(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

International Trade Centre. (2025). *Trade Map: Trade statistics for international business development*. https://www.trademap.org

Legiscomex. (2025a). *Estadísticas de comercio exterior*. https://www.legiscomex.com/informacion-estadisticas-de-comercio-exterior

Legiscomex. (2025b). *Desarrolle inteligencia de mercados a bajo costo*. https://www.legiscomex.com/Documentos/INTELIGENCIADEMERCADOS

McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt & J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Ministerio de Agricultura y Desarrollo Rural. (2025). *Agronet: Estadísticas agropecuarias*. https://www.agronet.gov.co

Montes Ninaquispe, J. C., Pantaleón Santa María, W. C., & Arbulú Ballesteros, M. A. (2025). Diversification and corporate strategy of agricultural products exports from a developing country. *Corporate and Business Strategy Review*. https://consensus.app/papers/details/5b262ae4810053c6a9193d2a32e44a23/

Serie de recursos — Inteligencia en Negocios Globales. (2026). *Métricas de comercio exterior e inteligencia de negocios globales. Documento 1 de 3 — Nivel básico: infraestructura de datos y métricas de posición comercial* [Documento de trabajo del curso].

The pandas development team. (2025). *pandas documentation* (Versión 2.x). https://pandas.pydata.org/docs/

Wickham, H. (2014). Tidy data. *Journal of Statistical Software, 59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10

World Bank. (2025). *World Development Indicators: Exports of goods and services (current US$), NE.EXP.GNFS.CD*. https://data.worldbank.org/indicator/NE.EXP.GNFS.CD

World Bank. (2025). *World Development Indicators: GDP (current US$), NY.GDP.MKTP.CD*. https://data.worldbank.org/indicator/NY.GDP.MKTP.CD

World Bank. (2025). *World Development Indicators: Imports of goods and services (current US$), NE.IMP.GNFS.CD*. https://data.worldbank.org/indicator/NE.IMP.GNFS.CD

---

*Cuaderno elaborado como material didáctico de la asignatura Inteligencia en Negocios Globales. Los cálculos son de elaboración propia a partir de datos públicos de Trade Map (ITC) y del Banco Mundial. El valor de producción usado en la sección 5 es un supuesto didáctico declarado y no debe emplearse en decisiones reales sin reemplazarlo por la cifra oficial correspondiente.*